In [0]:
# Silver layer: orders (B2B order facts)
# Source: b2b_ecommerce.bronze.orders (append-only, connector-managed,
#         already typed: long order_id, int ids, DATE order_date /
#         expected_delivery_date, decimal(8,2) shipping_cost, timestamps)
# Grain:  one row per order_id AND version (SCD Type 2); the current row
#         is the one with __END_AT IS NULL. Every bronze row is already
#         unique by order_id (11,223 rows = 11,223 ids), so no dedup is
#         needed - the AUTO CDC merge below is what keeps it unique if a
#         changed order is ever re-synced from the source.
# Profile (2026-11-12, 11,223 rows): status arrives in 5 real values but
#         344 rows carry case/trim variants (' delivered', 'Delivered',
#         'DELIVERED', ...) -> lower+trim standardizes all to canonical.
#         payment_method / shipping_method / channel are already clean
#         lowercase. 192 rows have NULL order_date and 214 rows have a
#         future order_date (up to 2027-12-08) - both KEPT and marked with
#         order_date_valid = false: silver preserves the business entity
#         (the order still counts for revenue by company, CLV and order
#         counts, and its order items still exist); excluding an invalid
#         date from a monthly trend is a GOLD reporting decision. 68 rows
#         have a negative shipping_cost (min -58.05) and 377 rows have
#         expected_delivery_date BEFORE order_date - both measures are
#         untrustworthy, so they are NULLified and the order itself kept.
#         2,224 rows legitimately have no po_number (kept NULL); all
#         present po_numbers match 'PO-<digits>'. FKs verified clean:
#         0 orphans vs current silver.customers and vs current
#         silver.companies of type 'buyer' (all 393 buyers are buyers).
# SCD:    11,212 of 11,223 rows have updated_at > created_at, i.e. orders
#         ARE updated at the source (status lifecycle pending -> confirmed
#         -> shipped -> delivered). A changed order re-synced into bronze
#         appends a new version under Type 2 and the history is kept (same
#         choice as silver.customers / companies / products / suppliers).
# Idempotency: AUTO CDC keyed by order_id MERGEs into b2b_ecommerce
#         .silver.orders - identical re-sends collapse (updated_at is
#         excluded from history tracking), re-ingestion never duplicates.
from pyspark import pipelines as dp
from pyspark.sql import functions as F

BRONZE_ORDERS = "b2b_ecommerce.bronze.orders"
# current dims live in the silver layer (same pipeline datasets)
SILVER_CUSTOMERS = "customers"
SILVER_COMPANIES = "companies"


def clean_text(name, lower=False):
    """Trim a text column; a blank value ("" or " ") becomes NULL, not ""."""
    trimmed = F.trim(F.col(name))
    if lower:
        trimmed = F.lower(trimmed)
    return (
        F.when(trimmed != "", trimmed)
        .otherwise(F.lit(None).cast("string"))
        .alias(name)
    )


# ---------------------------------------------------------------------------
# 1) Cleaned stream of bronze.orders (+ FK match flags vs current silver
#    customers / buyer companies)
# ---------------------------------------------------------------------------
@dp.temporary_view(name="orders_clean_stream")
# Data quality validation (only checks that can actually catch bad rows).
# Drop-level: without the merge key or the CDC sequence the row cannot
# be merged at all - quarantine it out of silver; the raw value stays in
# bronze for inspection. Everything else is kept and MARKED, never
# dropped - silver preserves business entities and flags problems, gold
# owns the reporting rules.
@dp.expect_or_drop("order_id_not_null", "order_id IS NOT NULL")
@dp.expect_or_drop("updated_at_not_null", "updated_at IS NOT NULL")
# Referential integrity: every order must reference a CURRENT silver
# customer and a CURRENT silver company of type 'buyer' - a missing link
# fails the whole update instead of silently landing in silver (hard stop
# is intended, same policy as silver.customers / suppliers; verified clean
# today: 0 orphans on both sides). No separate not-null warn checks: a
# NULL id never matches the join, so the exists checks already catch it.
@dp.expect_or_fail("customer_id_exists", "_customer_id_match IS NOT NULL")
@dp.expect_or_fail("buyer_company_id_exists", "_company_id_match IS NOT NULL")
# Warn-level: the row is kept and the problem only recorded in the event
# log (same policy as silver.customers email/phone checks). These fire
# only on NEW dirt from the source - all current rows pass after cleaning.
@dp.expect("status_valid", "status IN ('pending', 'confirmed', 'shipped', 'delivered', 'cancelled')")
@dp.expect("payment_method_valid", "payment_method IN ('credit_card', 'bank_transfer', 'cheque', 'cash_on_delivery')")
@dp.expect("shipping_method_valid", "shipping_method IN ('standard', 'express', 'pickup')")
@dp.expect("channel_valid", "channel IN ('web', 'email', 'phone', 'sales_rep')")
@dp.expect("po_number_format_valid", "po_number IS NULL OR po_number RLIKE '^PO-[0-9]+$'")
# Kept-but-invalid order dates (192 missing + 214 future) - counted here
# so they stay visible in the pipeline quality metrics; the row itself
# is kept, the problem is marked by the order_date_valid flag column.
@dp.expect("order_date_valid", "order_date_valid IS TRUE")
def orders_clean_stream():
    bronze = spark.readStream.table(BRONZE_ORDERS)

    # stream-static LEFT JOINs against CURRENT silver rows: a dim row that
    # exists in bronze but was dropped from silver does not match - the
    # row becomes an orphan and the fail-level expectation stops the
    # update. An expectation is a per-row column expression, so each FK
    # check needs its match flag as a column.
    customers = (
        spark.read.table(SILVER_CUSTOMERS)
        .filter("__END_AT IS NULL")
        .select(F.col("customer_id").alias("_customer_id_match"))
        .distinct()
    )
    buyer_companies = (
        spark.read.table(SILVER_COMPANIES)
        .filter("__END_AT IS NULL")
        .filter(F.col("company_type") == "buyer")
        .select(F.col("company_id").alias("_company_id_match"))
        .distinct()
    )

    bronze = bronze.join(
        customers,
        bronze["customer_id"] == customers["_customer_id_match"],
        "left",
    ).join(
        buyer_companies,
        bronze["buyer_company_id"] == buyer_companies["_company_id_match"],
        "left",
    )

    return bronze.select(
        F.col("order_id").cast("long").alias("order_id"),
        F.col("buyer_company_id").cast("int").alias("buyer_company_id"),
        F.col("customer_id").cast("int").alias("customer_id"),
        F.col("order_date").cast("date").alias("order_date"),
        # quality flag marking the kept orders whose date cannot be
        # trusted for time-based reporting: 192 missing + 214 future.
        # order_date itself is kept AS-IS (a future date may become valid
        # with time, NULL stays NULL) - gold decides report exclusions.
        (
            F.col("order_date").isNotNull()
            & (F.col("order_date") <= F.current_date())
        ).alias("order_date_valid"),
        # 344 rows arrive with case/trim variants -> canonical lowercase
        clean_text("status", lower=True),
        clean_text("payment_method", lower=True),
        clean_text("shipping_method", lower=True),
        # invalid measure is NULLified, the order row itself is kept:
        # 68 rows have a negative shipping_cost (min -58.05)
        F.when(F.col("shipping_cost") >= 0, F.col("shipping_cost"))
        .otherwise(F.lit(None))
        .cast("decimal(8,2)")
        .alias("shipping_cost"),
        # blank po_number becomes NULL (2,224 rows legitimately have
        # none); real po_numbers kept as-is (all match 'PO-<digits>')
        clean_text("po_number"),
        # 377 rows have a delivery date BEFORE the order date - the value
        # is untrustworthy, not the order: NULLify it and keep the row
        # (an order with a NULL order_date keeps its delivery date: it
        # cannot be judged against a missing order date)
        F.when(
            F.col("expected_delivery_date") < F.col("order_date"),
            F.lit(None),
        )
        .otherwise(F.col("expected_delivery_date"))
        .cast("date")
        .alias("expected_delivery_date"),
        clean_text("channel", lower=True),
        F.col("created_at").cast("timestamp").alias("created_at"),
        F.col("updated_at").cast("timestamp").alias("updated_at"),
        # internal FK flags, never land in silver
        F.col("_customer_id_match"),
        F.col("_company_id_match"),
    )


# ---------------------------------------------------------------------------
# 2) Silver target: b2b_ecommerce.silver.orders
#    Rows are MERGEd by order_id (SCD Type 2): identical re-sends update
#    in place, a genuinely changed row (e.g. status pending -> shipped)
#    opens a new version (__START_AT / __END_AT validity columns), the
#    history is kept.
# ---------------------------------------------------------------------------
dp.create_streaming_table(
    name="orders",
    comment=(
        "Silver orders: cleaned and standardized; one row per order_id and "
        "version (SCD Type 2 merge - latest updated_at wins, history kept)"
    ),
)

dp.create_auto_cdc_flow(
    target="orders",
    source="orders_clean_stream",
    keys=["order_id"],
    sequence_by="updated_at",
    stored_as_scd_type=2,
    # updated_at itself is excluded from history tracking: an identical
    # re-send (same values, newer updated_at) updates in place instead of
    # opening a duplicate version (same fix as silver.products).
    track_history_except_column_list=["updated_at"],
    # internal FK flags must not land in the target table
    except_column_list=["_customer_id_match", "_company_id_match"],
)